In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

In [ ]:
load_dotenv()
api_key=os.getenv("OPENAI_API_KEY")
client=OpenAI(api_key=api_key)

### Vector Store Creation

In [ ]:
vector_store = client.vector_stores.create(name="Industrial_taxonomy_space")
taxonomy_vector_id=vector_store.id
print(taxonomy_vector_id)

vs_692f1a57283c819186a1b1319aabce42


### Uploading to vector store

In [ ]:
vector_store_id = taxonomy_vector_id

# Path to your single JSON file
file_path = "/content/article1 emotion taxanomy.json"

print(f"Uploading {file_path}...")

# Upload the single file
with open(file_path, "rb") as f:
    file_response = client.vector_stores.files.upload_and_poll(
        vector_store_id=vector_store_id,
        file=f
    )

# Check status (optional)
print(f"Uploaded file: Status - {file_response.status}")

print("Upload completed successfully!")


Uploading /content/article1 emotion taxanomy.json...
Uploaded file: Status - completed
Upload completed successfully!


In [ ]:
article_text = """ HCA Healthcare is one of the leading hospital chains in the U.S., with a significant presence across the country, although its business is concentrated primarily in Texas and Florida.\n\nThe industry is competitive, but HCA consistently performs well nonetheless. The company has several competitive advantages, including its extensive network of facilities that span acute care hospitals, specialty care, urgent care, and more. This diversified mix of facilities enables it to retain patients within its ecosystem, granting it significant negotiating power with private insurers and government payers.\n\nAs a result, HCA Healthcare has built relationships with these third-party institutions. Further, HCA routinely invests in cutting-edge technology, another factor that allows it to attract patients. These dynamics have enabled the company to increase its market share over time. It had a 24% market share in 2012, and 10 years later, that had risen to 27%.\n\nHCA also records consistent revenue and profits. In the second quarter, the company's top line reached $18.6 billion, representing a 6.4% increase compared to the same period last year. The company's earnings per share climbed 23.5% year over year to $6.83.\n\nOver the next decade, HCA Healthcare should benefit from increased hospital spending. In the meantime, the company will continue to implement the same strategy that has made it so successful over the years. It's also worth noting that there are significant regulatory and financial barriers to entry in the industry.\n\nNewcomers then have to compete with established corporations to attract patients and third-party payers, but it's challenging to do so because those, like HCA Healthcare, that have been around for a while have already built solid relationships with major players in the healthcare sector. That's why it's hard to imagine HCA Healthcare completely losing its position. The stock should continue performing well over the long run."


"""

### Retrieval from Vector store

In [ ]:
import json

# -----------------------------
# Load your emotion + sentiment taxonomy JSON
# -----------------------------
taxonomy_file = "/content/article1 emotion taxanomy.json"

with open(taxonomy_file, "r") as f:
    taxonomy = json.load(f)

# -----------------------------
# Build a lookup:
# Industry -> Sub-category -> {Emotions, Sentiments}
# -----------------------------
industry_lookup = {}

for entry in taxonomy:
    industry_name = entry["Industry"]
    sub_lookup = {}

    for sub in entry.get("Sub-categories", []):
        sub_name = sub["Name"]

        # Sentiment block exists like:
        # "Sentiments": {
        #     "Positive": [...],
        #     "Neutral": [...],
        #     "Negative": [...]
        # }
        sentiments = sub.get("Sentiments", {
            "Positive": [],
            "Neutral": [],
            "Negative": []
        })

        sub_lookup[sub_name] = {

            "Sentiments": sentiments
        }

    industry_lookup[industry_name] = sub_lookup



# -----------------------------
# RAG SEARCH
# -----------------------------
results = client.vector_stores.search(
    vector_store_id="vs_6914573daa5c81918616d8a8da7351d4",
    query=article_text,
)

# -----------------------------
# Enrich results with Emotions + Sentiments
# -----------------------------
for item in results.data:
    try:
        content_json = json.loads(item.content[0].text)

        industry = content_json.get("Industry", "")
        sub_category = content_json.get("Sub-category", "")

        # Safely fetch taxonomy values
        taxonomy_entry = industry_lookup.get(industry, {}).get(sub_category, {})


        sentiments = taxonomy_entry.get("Sentiments", {
            "Positive": [],
            "Neutral": [],
            "Negative": []
        })

        # Attach both to the final output

        content_json["Sentiments"] = sentiments

        # Print enriched result
        print(f"Score: {item.score:.3f}")
        print(json.dumps(content_json, indent=2))
        print("...\n")

    except json.JSONDecodeError:
        print(f"Score: {item.score:.3f}")
        print(item.content[0].text[:300], "...\n")


Score: 0.840
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Medical Devices",
  "Description": "Diagnostic and therapeutic devices for hospital and home use.",
  "Sentiments": {
    "Positive": [],
    "Neutral": [],
    "Negative": []
  }
}
...

Score: 0.839
{
  "Industry": "Healthcare & Biotech",
  "Sub-category": "Digital Health",
  "Description": "Software-enabled care delivery, monitoring, and patient engagement.",
  "Sentiments": [
    {
      "Type": "Positive",
      "Mood": "Bullish",
      "Intensity": "Strong",
      "Trend": "Improving",
      "Magnitude": "Large",
      "Time Horizon": "Long-term",
      "Confidence": "High",
      "Typical Investor Action": "Small Buy / Tactical entry",
      "Examples": [
        "Strong adoption of telemedicine",
        "AI-healthtech partnerships gaining traction",
        "Successful product trials or regulatory approvals"
      ],
      "Explanation": "Rapid adoption of digital health solutions and strong innovation tren

In [ ]:
results_sorted = sorted(results.data, key=lambda x: x.score, reverse=True)

### Formatting Vector Retrieval

In [ ]:
def format_results(results):
    """
    Takes vector store search results and returns a formatted string
    containing Industry, Sub-category, Emotions, and Sentiments,
    suitable to feed into the LLM.
    """
    formatted_lines = []

    for item in results:
        for part in item.content:
            try:
                data = json.loads(part.text)
                industry = data.get("Industry")
                sub_category = data.get("Sub-category")

                if industry and sub_category:

                    # -----------------------------
                    # GET TAXONOMY (Emotions + Sentiments)
                    # -----------------------------
                    taxonomy_entry = industry_lookup.get(industry, {}).get(sub_category, {})

                    # Emotions
                    #emotions_list = taxonomy_entry.get("Emotions", [])
                    #emotions_str = (
                       # ", ".join([e.get("Name", "") for e in emotions_list])
                        #if emotions_list else "No emotions found"
                    #)

                    # Sentiments
                    # Sentiments handling
                    positive = neutral = negative = "None"
                    for s in taxonomy_entry.get("Sentiments", []):
                          if s.get("Type") == "Positive":
                              positive = s
                          elif s.get("Type") == "Neutral":
                              neutral = s
                          elif s.get("Type") == "Negative":
                              negative = s


                    # -----------------------------
                    # FORMAT OUTPUT LINE
                    # -----------------------------
                    formatted_lines.append(
                        f"- {industry} (Sub-category: {sub_category}) | "
                        #f"Emotions: {emotions_str} | "
                        f"Sentiments → Positive: [{positive}] | Neutral: [{neutral}] | Negative: [{negative}]"
                    )

            except json.JSONDecodeError:
                continue

    return "\n".join(formatted_lines)


# Usage example
formatted_results = format_results(results_sorted)
print("Candidates to feed into LLM:\n", formatted_results)


Candidates to feed into LLM:
 - Healthcare & Biotech (Sub-category: Medical Devices) | Sentiments → Positive: [None] | Neutral: [None] | Negative: [None]
- Healthcare & Biotech (Sub-category: Digital Health) | Sentiments → Positive: [{'Type': 'Positive', 'Mood': 'Bullish', 'Intensity': 'Strong', 'Trend': 'Improving', 'Magnitude': 'Large', 'Time Horizon': 'Long-term', 'Confidence': 'High', 'Typical Investor Action': 'Small Buy / Tactical entry', 'Examples': ['Strong adoption of telemedicine', 'AI-healthtech partnerships gaining traction', 'Successful product trials or regulatory approvals'], 'Explanation': 'Rapid adoption of digital health solutions and strong innovation trends indicate a very positive long-term outlook.'}] | Neutral: [{'Type': 'Neutral', 'Mood': 'Balanced', 'Intensity': 'Mild', 'Trend': 'Stable', 'Magnitude': 'Small', 'Time Horizon': 'Short-term', 'Confidence': 'Medium', 'Typical Investor Action': 'Hold / Monitor adoption trends', 'Examples': ['Slow but steady user gro

### System Prompt

In [ ]:
SYSTEM_PROMPT = f"""
You are an industry-aware  Sentiment Classification Model.

Your job:
Given a news article AND the taxonomy evidence (industry → sub-category → sentiment definitions), classify the SINGLE most central sentiment expressed toward the already-identified industry and sub-category.

You DO NOT identify the industry.
Your ONLY tasks are:
1. Select the correct sentiment  label (positive, neutral,negative) from the taxonomy.
2. Select Mood,
      Intensity
      Trend
      Magnitude
      "Time Horizon
      "Confidence": "Any",
      "Typical Investor Action": [
        "Hold / Wait for clearer signal" values exactly as defined in the taxonomy.
3. Produce the final output mapped to the correct industry + sub-category.
4. Provide a short reason.
5. **Also classify sentiment using the sentiment taxonomy.**
6. **Return sentimentin the final JSON output, using the full sentiment structure:**

Sentiments must include:
- Type: Positive / Neutral / Negative
- Mood
- Intensity
- Trend
- Magnitude
- Time Horizon
- Confidence
- Typical Investor Action

Strict Rules & Behavior Requirements:

────────────────────────────────────────────
【1. Use ONLY Taxonomy Evidence】
────────────────────────────────────────────
- You MUST use the exact strings for:
  • Industry
  • Sub-category
  • Sentiment Type, Mood, and all other sentiment fields
- You may not invent new labels, intensities, sentiments, or sub-categories.
- **Sentiments MUST be selected ONLY from taxonomy and follow the full structure.**

────────────────────────────────────────────
【4. Reasoning Rules】
────────────────────────────────────────────
- Keep the reason very short and factual.
- If the sentiment is supported by retrieved taxonomy evidence:
    End the reason with: **Retrieved from vector store**
-


────────────────────────────────────────────
【5A. Sentiment Logic (ADDED)】
────────────────────────────────────────────
- Use ONLY the sentiment taxonomy.
- Map article tone to the closest sentiment Type (Positive / Neutral / Negative).
- Populate all fields of the sentiment structure exactly as defined:
  • Mood
  • Intensity
  • Trend
  • Magnitude
  • Time Horizon
  • Confidence
  • Typical Investor Action
- The sentiment structure must appear under each shortlisted emotion.
- Do NOT invent additional fields.

────────────────────────────────────────────
【6. Hallucination Guardrails】
────────────────────────────────────────────
- Use ONLY article text + taxonomy evidence.
- Do NOT invent events, numbers, partnerships, technologies, or industries.
- Do NOT overfit the article — classify only what is clearly supported.

────────────────────────────────────────────
【7. Output Requirements】
────────────────────────────────────────────
Output a JSON object in the exact structure:

{{
  "shortlisted_emotions": [
    {{
      "industry": "<Industry from taxonomy>",
      "sub_category": "<Sub-category from taxonomy>",
       "reason": "<Short explanation ending with either 'Retrieved from vector store'
      "Sentiments": [
        {{
          "Type": "<Positive/Neutral/Negative>",
          "Mood": "<Mood description>",
          "Intensity": "<Mild / Strong / Array>",
          "Trend": "<Improving / Stable / Deteriorating / Array>",
          "Magnitude": "<Small / Large / Array>",
          "Time Horizon": "<Short-term / Long-term / Both / Array>",
          "Confidence": "<Medium / High / Any / Array>",
          "Typical Investor Action": ["<Action 1>", "<Action 2>", "..."]
          "Explanation": "<Short explanation ending with either 'Retrieved from vector store'>"
        }}
      ]
    }}
  ],
  "emergent_emotion_candidate": null
}}
"""


### User Prompt

In [ ]:
USER_PROMPT = f"""
Article:
{article_text}

Retrieved Taxonomy Evidence:
{formatted_results}

Task:
- Shortlist  full sentiment taxonomy relevant to the industry and sub-category found in the retrieved evidence.
- For each shortlisted item return:
    industry,
    sub_category,
    reason,
    Sentiments: [
        {{
            Type (Positive / Neutral / Negative),
            Mood,
            Intensity (Mild / Strong / Array),
            Trend (Improving / Stable / Deteriorating / Array),
            Magnitude (Small / Large / Array),
            Time Horizon (Short-term / Long-term / Both / Array),
            Confidence (Medium / High / Any / Array),
            Typical Investor Action (array of actions)
            Explanation <Short explanation ending with either 'Retrieved from vector store'>"
        }}
    ]

- Always take item  directly from the evidence, the reason MUST end with: "Retrieved from vector store".
- Answer deterministically. Identical input must always yield identical output.
- Use only the article text and the provided taxonomy; do not hallucinate events, numbers, sub-categories, or companies.
- Do not add anything that is not in taxonomy; only retrieve sub-categories from taxonomy.
- Return **at least 2 industries** with  corresponding sentiments.

"""


### JSON Schema

In [ ]:
text_format = {
    "format": {
        "type": "json_schema",
        "name": "SentimentClassification",
        "schema": {
            "type": "object",
            "properties": {
                "shortlisted_sentiments": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "industry": {"type": "string"},
                            "sub_category": {"type": "string"},
                            "Sentiments": {
                                "type": "array",
                                "items": {
                                    "type": "object",
                                    "properties": {
                                        "Type": {"type": "string", "enum": ["Positive", "Neutral", "Negative"]},
                                        "Mood": {"type": "string"},
                                        "Intensity": {"type": "string"},
                                        "Trend": {"type": "string"},
                                        "Magnitude": {"type": "string"},
                                        "Time Horizon": {"type": "string"},
                                        "Confidence": {"type": "string"},
                                        "Typical Investor Action": {"type": "string"},
                                        "Explanation": {"type": "string"}
                                    },
                                    "required": [
                                        "Type",
                                        "Mood",
                                        "Intensity",
                                        "Trend",
                                        "Magnitude",
                                        "Time Horizon",
                                        "Confidence",
                                        "Typical Investor Action",
                                        "Explanation"
                                    ],
                                    "additionalProperties": False
                                }
                            }
                        },
                        "required": ["industry", "sub_category", "Sentiments"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["shortlisted_sentiments"],
            "additionalProperties": False
        }
    }
}


### LLM Call

In [ ]:
response = client.responses.create(
        model="gpt-5-mini",
        input=[
            {"role": "system", "content": SYSTEM_PROMPT },
            {"role": "user", "content": USER_PROMPT}
        ],
        text=text_format,

)
result=response.output_text
print(result)


{
  "shortlisted_sentiments": [
    {
      "industry": "Healthcare & Biotech",
      "sub_category": "Digital Health",
      "Sentiments": [
        {
          "Type": "Positive",
          "Mood": "Bullish",
          "Intensity": "Strong",
          "Trend": "Improving",
          "Magnitude": "Large",
          "Time Horizon": "Long-term",
          "Confidence": "High",
          "Typical Investor Action": "Small Buy / Tactical entry",
          "Explanation": "Rapid adoption of digital health solutions and strong innovation trends indicate a very positive long-term outlook. Retrieved from vector store"
        }
      ]
    },
    {
      "industry": "Healthcare & Biotech",
      "sub_category": "Health Insurance",
      "Sentiments": [
        {
          "Type": "Positive",
          "Mood": "Optimistic",
          "Intensity": "Mild",
          "Trend": "Improving",
          "Magnitude": "Small",
          "Time Horizon": "Short-term",
          "Confidence": "Medium",
     

In [ ]:
try:
    parsed = json.loads(result)

    pretty_json = json.dumps(parsed, indent=4, ensure_ascii=False)

    print(pretty_json)

except json.JSONDecodeError as e:
    print("Invalid JSON:", e)

{
    "shortlisted_sentiments": [
        {
            "industry": "Healthcare & Biotech",
            "sub_category": "Digital Health",
            "Sentiments": [
                {
                    "Type": "Positive",
                    "Mood": "Bullish",
                    "Intensity": "Strong",
                    "Trend": "Improving",
                    "Magnitude": "Large",
                    "Time Horizon": "Long-term",
                    "Confidence": "High",
                    "Typical Investor Action": "Small Buy / Tactical entry",
                    "Explanation": "Rapid adoption of digital health solutions and strong innovation trends indicate a very positive long-term outlook. Retrieved from vector store"
                }
            ]
        },
        {
            "industry": "Healthcare & Biotech",
            "sub_category": "Health Insurance",
            "Sentiments": [
                {
                    "Type": "Positive",
                    "Mood"

In [ ]:
# Token Usage
print("\n--- TOKEN USAGE ---")
print("Input tokens:", response.usage.input_tokens)
print("Output tokens:", response.usage.output_tokens)



--- TOKEN USAGE ---


NameError: name 'response' is not defined